In [23]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras import layers, models

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# image size and batch size
image_size = (224, 224,3)                   #height,width,color channels
batch_size = 16

# Directory for the training set
train_dir = 'C:/Users/prajw/OneDrive/Documents/Desktop/MINI PROJECT ONCONOVA/melanoma_cancer_dataset/train'
                                                        #ADD PATH 
# Directory for the testing set
test_dir = "C:/Users/prajw/OneDrive/Documents/Desktop/MINI PROJECT ONCONOVA/melanoma_cancer_dataset/test"

# ImageDataGenerator for rescaling and image augmentation (helps reduce overfitting)
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2,                # 20% of the training data will be used for validation
                                    rotation_range=10,                                  # Randomly rotate images
                                    width_shift_range=0.2,                              # Randomly shift images horizontally
                                    height_shift_range=0.2,                             # Randomly shift images vertically
                                    shear_range=0.2,                                    # (basically sliding image)
                                    zoom_range=0.2,                                     # Zoom in/out on images
                                    horizontal_flip=True,                               # Flip images horizontally
                                    brightness_range=[0.8, 1.2],                        # Added brightness adjustment
                                    fill_mode='nearest')              

# Load training data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size[:2],  
    batch_size=batch_size,
    class_mode='binary',                 #only 2 output classes
    subset='training'                    #this is for the actual training data (80%)
)

# Load validation data
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size[:2],  # Updated to use 'image_size'
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'  # Use this for the validation data (20%)
)

Found 7684 images belonging to 2 classes.
Found 1921 images belonging to 2 classes.


In [25]:
from tensorflow.keras.applications import EfficientNetB0
model=EfficientNetB0(include_top=False,weights='imagenet',input_shape=image_size)        #loading efficientnet without last layer and pre trained weights
model.trainable=False                                            #freezing the base model
model.summary()

Model: "efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,049,571 (15.45 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,049,571 (15.45 MB)

In [32]:
#building the output layer
global_avg_pool = layers.GlobalAveragePooling2D()(model.output)                              #This layer takes the maximum value from each feature map
#Note: since global_max_pool results in 1-D structure , skipped flatten layer


output_avg_pool = layers.Dense(1, activation='sigmoid', name='avg_pool_output')(global_avg_pool)
#Dense to create output layer, 1 refers to single neuron,'sigmoid'(best for binary classification)
 
avg_model=models.Model(inputs=model.input, outputs=output_avg_pool)                # Model is from keras and creates a model taking input and output architecture of cnn
#'model.input' helps make sure shape of our layers and efficient net layers are same 

In [33]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.metrics import Precision, Recall

# Compile the max model
optimizer = Adam(learning_rate=1e-3)  # Use a higher learning rate for the initial training
avg_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy', Precision(), Recall()])

In [39]:
from sklearn.utils import class_weight
import numpy as np

# Get class weights from training data
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights_dict = dict(enumerate(class_weights))
cnn = avg_model.fit(
    train_generator,
    epochs=7,
    validation_data=validation_generator,
    verbose=1                                            # output control
)

Epoch 1/7
481/481 ━━━━━━━━━━━━━━━━━━━━ 209s 433ms/step - accuracy: 0.5156 - loss: 0.6942 - precision_4: 0.4903 - recall_4: 0.2942 - val_accuracy: 0.5206 - val_loss: 0.6927 - val_precision_4: 0.0000e+00 - val_recall_4: 0.0000e+00
Epoch 2/7
481/481 ━━━━━━━━━━━━━━━━━━━━ 206s 428ms/step - accuracy: 0.5193 - loss: 0.6934 - precision_4: 0.4680 - recall_4: 0.0489 - val_accuracy: 0.5206 - val_loss: 0.6923 - val_precision_4: 0.0000e+00 - val_recall_4: 0.0000e+00
Epoch 3/7
481/481 ━━━━━━━━━━━━━━━━━━━━ 208s 433ms/step - accuracy: 0.5129 - loss: 0.6924 - precision_4: 0.4432 - recall_4: 0.0792 - val_accuracy: 0.5206 - val_loss: 0.6923 - val_precision_4: 0.0000e+00 - val_recall_4: 0.0000e+00
Epoch 4/7
481/481 ━━━━━━━━━━━━━━━━━━━━ 206s 429ms/step - accuracy: 0.5141 - loss: 0.6934 - precision_4: 0.3425 - recall_4: 0.0093 - val_accuracy: 0.5206 - val_loss: 0.6923 - val_precision_4: 0.0000e+00 - val_recall_4: 0.0000e+00
Epoch 5/7
481/481 ━━━━━━━━━━━━━━━━━━━━ 208s 432ms/step - accuracy: 0.5266 - loss: 0.

In [35]:
for layer in model.layers[-20:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

In [37]:
global_avg_pool = layers.GlobalAveragePooling2D()(model.output)
dropout = layers.Dropout(0.5)(global_avg_pool)  # Dropout added
output_avg_pool = layers.Dense(1, activation='sigmoid', name='avg_pool_output')(dropout)
   

avg_model = models.Model(inputs=model.input, outputs=output_avg_pool)

# Optimizer: AdamW with weight decay
optimizer = Adam(learning_rate=1e-4)

# Compile the model with additional metrics
avg_model.compile(optimizer=optimizer, loss='binary_crossentropy',
                  metrics=['accuracy', Precision(), Recall()])

# Learning rate scheduling and early stopping
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [38]:
cnn = avg_model.fit(
    train_generator,
    epochs=25,
    validation_data=validation_generator,
    callbacks=[lr_scheduler, early_stopping],
    verbose=1
)

Epoch 1/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 227s 438ms/step - accuracy: 0.5167 - loss: 0.6967 - precision_4: 0.4800 - recall_4: 0.3130 - val_accuracy: 0.4794 - val_loss: 0.6981 - val_precision_4: 0.4794 - val_recall_4: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/25
246/481 ━━━━━━━━━━━━━━━━━━━━ 3:47 969ms/step - accuracy: 0.4907 - loss: 0.6964 - precision_4: 0.4729 - recall_4: 0.4452

KeyboardInterrupt: 